# Baseline results: Compare Gemini 2 Flash, Gemini 2.5 Flash, GPT-4.1

Load evaluation results from three file-search baseline runs and compare:

- **Gemini 2.0 Flash**: `experiment-scripts/baselines_gemini_file_search/Gemini-2.0/`
- **Gemini 2.5 Flash**: `experiment-scripts/baselines_gemini_file_search/Gemini-2.5/`
- **GPT-4.1**: `experiment-scripts/baselines_openai_file_search/GPT-4.1/`

Reports:
- **By PDF**: overall correctness, completeness, overall score — compared across the three models
- **By category**: exact_match, numeric_tolerance, structured_text — aggregated and per-PDF, by model

In [ ]:
import json
from pathlib import Path
from typing import Optional
import pandas as pd

# Resolve experiment-scripts directory (run from repo root or experiment-analysis/)
_candidates = [
    Path.cwd() / "experiment-scripts",
    (Path.cwd() / ".." / "experiment-scripts").resolve(),
    Path.cwd(),
]
SCRIPT_BASE = next((p for p in _candidates if p.exists()), None)
if SCRIPT_BASE is None:
    raise FileNotFoundError("Could not find experiment-scripts. Run from repo root or experiment-analysis/.")

# Three baseline runs to compare
BASELINE_RUNS = [
    ("Gemini 2.0 Flash", SCRIPT_BASE / "baselines_gemini_file_search" / "Gemini-2.0"),
    ("Gemini 2.5 Flash", SCRIPT_BASE / "baselines_gemini_file_search" / "Gemini-2.5"),
    ("GPT-4.1", SCRIPT_BASE / "baselines_openai_file_search" / "GPT-4.1"),
]

def load_summary_metrics(trial_dir: Path) -> Optional[dict]:
    path = trial_dir / "evaluation" / "summary_metrics.json"
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

for label, path in BASELINE_RUNS:
    n = len([d for d in path.iterdir() if d.is_dir()]) if path.exists() else 0
    print(f"  {label}: {path.name} ({n} trials)")

In [ ]:
# Load summary metrics for all three runs
rows_by_pdf = []
rows_by_category = []

for model_label, run_path in BASELINE_RUNS:
    if not run_path.exists():
        continue
    trial_dirs = sorted([d for d in run_path.iterdir() if d.is_dir()])
    for trial_dir in trial_dirs:
        data = load_summary_metrics(trial_dir)
        if data is None:
            continue
        pdf_name = trial_dir.name
        overall = data.get("overall", {})
        rows_by_pdf.append({
            "model": model_label,
            "pdf": pdf_name,
            "avg_correctness": overall.get("avg_correctness"),
            "avg_completeness": overall.get("avg_completeness"),
            "avg_overall": overall.get("avg_overall"),
            "total_columns": overall.get("total_columns"),
        })
        for cat_name, cat_metrics in data.get("by_category", {}).items():
            rows_by_category.append({
                "model": model_label,
                "pdf": pdf_name,
                "category": cat_name,
                "avg_correctness": cat_metrics.get("avg_correctness"),
                "avg_completeness": cat_metrics.get("avg_completeness"),
                "avg_overall": cat_metrics.get("avg_overall"),
                "column_count": cat_metrics.get("column_count"),
            })

df_by_pdf = pd.DataFrame(rows_by_pdf)
df_by_category = pd.DataFrame(rows_by_category)
print(f"Loaded {len(df_by_pdf)} rows (pdf × model), {len(df_by_category)} category rows")

## By PDF: overall correctness, completeness, overall score (compare models)

In [ ]:
# Pivot: one row per PDF, columns = model overall score (%)
pivot_overall_by_pdf = df_by_pdf.pivot(index="pdf", columns="model", values="avg_overall")
(pivot_overall_by_pdf * 100).round(1)

In [ ]:
# Correctness and completeness by PDF (columns = model)
pivot_correctness_by_pdf = df_by_pdf.pivot(index="pdf", columns="model", values="avg_correctness")
pivot_completeness_by_pdf = df_by_pdf.pivot(index="pdf", columns="model", values="avg_completeness")
print("Avg correctness by PDF × model (0–100):")
display((pivot_correctness_by_pdf * 100).round(1))
print("Avg completeness by PDF × model (0–100):")
display((pivot_completeness_by_pdf * 100).round(1))

## By category: exact_match, numeric_tolerance, structured_text (aggregated per model)

In [ ]:
agg = df_by_category.groupby("category").agg(
    avg_correctness=("avg_correctness", "mean"),
    avg_completeness=("avg_completeness", "mean"),
    avg_overall=("avg_overall", "mean"),
    total_column_count=("column_count", "sum"),
).reset_index()
agg_display = agg.copy()
for col in ["avg_correctness", "avg_completeness", "avg_overall"]:
    agg_display[col] = (agg_display[col] * 100).round(1).astype(str) + "%"
agg_display

In [ ]:
# Correctness and completeness by category, per model (from df_by_category)
pivot_cat_correctness = df_by_category.pivot_table(
    index="category", columns="model", values="avg_correctness", aggfunc="mean"
)
pivot_cat_completeness = df_by_category.pivot_table(
    index="category", columns="model", values="avg_completeness", aggfunc="mean"
)
print("Avg correctness by category × model (0–100):")
display((pivot_cat_correctness * 100).round(1))
print("Avg completeness by category × model (0–100):")
display((pivot_cat_completeness * 100).round(1))

## By category per PDF: overall score (PDF × category × model)

In [ ]:
# Pivot: pdf × (model, category) — show overall score per PDF per model per category
# MultiIndex columns: (model, category)
pivot_pdf_cat_model = df_by_category.pivot_table(
    index="pdf", columns=["model", "category"], values="avg_overall"
)
(pivot_pdf_cat_model * 100).round(1)

## Per model: PDF × category overall score

In [ ]:
# For each model, show PDF × category (overall score %)
for model in df_by_category["model"].unique():
    sub = df_by_category[df_by_category["model"] == model]
    p = sub.pivot(index="pdf", columns="category", values="avg_overall")
    print(f"--- {model} ---")
    display((p * 100).round(1))